# Clustering deep-sea coral communities by taxon

We group survey events by the relative abundances of recorded coral families. The resulting labels describe patterns in the observations and provide an input for Issue #11.

**Setup.** Install the repository requirements and `scikit-learn` in the notebook kernel. Run `ingest.ipynb` and `clean.ipynb` first, or use the team's matching Parquet snapshot. This notebook also needs a NOAA metadata extract keyed by catalog number. It downloads and caches the extract on first run. Preserve both inputs together to reproduce the analysis. The notebook runs from the repository root or the `notebooks` directory.

We adapt the chord transform, Ward linkage, and K-Means refinement in [Sigler et al. (2023), section 2.4.1](https://doi.org/10.3354/meps14317) to event-level NOAA observation counts. Counts are proxies for observed abundance because survey areas and effort vary.

In [1]:
from pathlib import Path
from tempfile import NamedTemporaryFile

import numpy as np
import pandas as pd
import requests
from scipy.stats import chi2_contingency
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_samples, silhouette_score
from sklearn.preprocessing import normalize
from threadpoolctl import threadpool_limits

repo_dir = Path.cwd()
if repo_dir.name == "notebooks":
    repo_dir = repo_dir.parent

data_dir = repo_dir / "data"
cleaned_path = data_dir / "processed" / "se_coral_records_cleaned.parquet"
interim_path = data_dir / "interim" / "se_coral_records.parquet"
input_path = cleaned_path if cleaned_path.is_file() else interim_path

if not input_path.is_file():
    raise FileNotFoundError("Run ingest.ipynb or place the team Parquet snapshot under data/.")

records = pd.read_parquet(input_path)
print(f"Input: {input_path.relative_to(repo_dir)} ({len(records):,} records)")

if input_path == interim_path:
    observation_dates = pd.to_datetime(records["obs_date"], format="%Y-%m-%d", errors="coerce")
    records = records.assign(obs_date=observation_dates).dropna(subset=["obs_date"])
    print(f"Records with valid dates: {len(records):,}")

Input: data\processed\se_coral_records_cleaned.parquet (71,499 records)


## Identify coral records and sampling metadata

The source includes corals, sponges, and fish. `ingest.ipynb` already requests the data provider and sampling equipment. We join NOAA's taxonomic category and dataset ID to the Parquet file using the unique NOAA catalog number. The metadata request uses the same regional, year, and deep-water filters as `ingest.ipynb`. The join checks that every input record has a metadata match before selecting the coral categories. The [NOAA ERDDAP data dictionary](https://www.ncei.noaa.gov/erddap/tabledap/deep_sea_corals.html) documents the source fields.

In [2]:
metadata_path = data_dir / "raw" / "se_coral_taxon_metadata.csv"
metadata_columns = {
    "CatalogNumber": "catalog_num",
    "DatasetID": "dataset_id",
    "DatabaseVersion": "database_version",
    "VernacularNameCategory": "taxon_category",
}
if not metadata_path.is_file():
    query_columns = ",".join(metadata_columns)
    ecosystems = "%28Southeast%20U.S.%20Continental%20Shelf%7CGulf%20of%20Mexico%7CCaribbean%20Sea%29"
    metadata_url = (
        "https://www.ncei.noaa.gov/erddap/tabledap/deep_sea_corals.csvp?"
        f"{query_columns}&ShallowFlag=0&ObservationYear%3E=2018&Country=%22USA%22"
        f"&LargeMarineEcosystem=~%22{ecosystems}%22"
    )
    metadata_path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = None
    try:
        with requests.get(metadata_url, stream=True, timeout=(20, 180)) as response:
            response.raise_for_status()
            with NamedTemporaryFile(dir=metadata_path.parent, delete=False) as temp_file:
                temporary_path = Path(temp_file.name)
                temp_file.writelines(response.iter_content(1 << 20))
        temporary_path.replace(metadata_path)
    finally:
        if temporary_path is not None:
            temporary_path.unlink(missing_ok=True)

metadata = pd.read_csv(
    metadata_path,
    usecols=list(metadata_columns),
    dtype={"CatalogNumber": "int64"},
    keep_default_na=False,
).rename(columns=metadata_columns)
matched_records = records.merge(metadata, on="catalog_num", how="left", validate="one_to_one", indicator=True)
unmatched_count = matched_records["_merge"].ne("both").sum()
if unmatched_count:
    raise ValueError(f"{unmatched_count:,} records lack catalog matches. Use matching NOAA snapshots.")
matched_records = matched_records.drop(columns="_merge")
versions = ", ".join(sorted(matched_records["database_version"].unique()))
print(f"NOAA database version in matched metadata: {versions}")

NOAA database version in matched metadata: 20260416-1


## Build the event-by-family matrix

Keep coral and sea pen observations with an identified family and a usable survey/event key. Pool observations by event and retain source IDs for provenance. Sum individual counts for each family in each event. An observation without a positive count contributes one recorded individual as a minimum-count proxy. We check that choice by repeating the analysis with only positive counts. Density is not pooled because it is sparse and the sampled area and effort vary.

Keep families recorded in at least five events for fitting. An event that has no abundance after filtering cannot be chord-normalized and is excluded from the revised fit. Preserve the full binary family matrix on those retained rows for composition plots and richness comparisons.

In [3]:
coral_categories = {
    "alcyonacean (unspecified)", "black coral", "gorgonian coral", "lace coral",
    "scleralcyonacea (unspecified)", "sea pen", "soft coral", "stony coral (branching)",
    "stony coral (cup coral)", "stony coral (unspecified)",
}
event_keys = ["survey_id", "event_id"]
missing_identifiers = {"", "-1", "na", "n/a", "none", "unknown", "null", "nan", "not provided"}
corals = matched_records.loc[matched_records["taxon_category"].isin(coral_categories)].copy()

for column in [*event_keys, "family"]:
    values = corals[column].astype("string").str.strip()
    corals[column] = values.mask(values.str.casefold().isin(missing_identifiers))

corals = corals.dropna(subset=[*event_keys, "family"])
if corals.empty:
    raise ValueError("No coral records have both a usable family and an event key.")

event_families = corals.groupby([*event_keys, "family"]).size().unstack(fill_value=0)
family_presence = event_families.gt(0).astype("uint8").sort_index()
family_richness = family_presence.sum(axis=1).rename("recorded_families")

print(f"Coral records with family and event key: {len(corals):,}")
print(f"Positive count coverage: {corals['count'].gt(0).mean():.1%}")
print(f"Density coverage: {corals['density'].notna().mean():.1%}")
print(f"Missing or nonpositive counts: {(~corals['count'].gt(0).fillna(False)).sum():,}")

minimum_events = 5
seed = 42

def abundance_matrix(observations, min_events):
    counts = observations["count"].where(observations["count"].gt(0), 1)
    matrix = observations.assign(abundance=counts).groupby([*event_keys, "family"])["abundance"].sum()
    matrix = matrix.unstack(fill_value=0).sort_index()
    matrix = matrix.loc[:, matrix.gt(0).sum().ge(min_events)]
    return matrix.loc[matrix.sum(axis=1).gt(0)]

abundance = abundance_matrix(corals, minimum_events)
features = normalize(abundance.to_numpy(dtype=float), norm="l2")
max_clusters = min(8, len(features) - 1, len(abundance.drop_duplicates()))
if max_clusters < 2:
    raise ValueError("At least three eligible events and two distinct abundance profiles are required.")
print(f"Fitted matrix: {len(abundance)} events × {abundance.shape[1]} families")
print(f"Events excluded by rare-family filtering: {len(family_presence) - len(abundance)}")

Coral records with family and event key: 46,364
Positive count coverage: 92.4%
Density coverage: 6.8%
Missing or nonpositive counts: 3,521
Fitted matrix: 272 events × 29 families
Events excluded by rare-family filtering: 3


## Ward clustering and K-Means refinement

The chord transform gives each event a unit-length abundance vector. Euclidean distances then compare taxon composition without directly reflecting the total number of counted individuals. For each candidate k, use Ward linkage, then seed K-Means with the Ward cluster means.

For indicator values, specificity is a cluster's mean relative abundance for a family divided by the sum of its mean relative abundances across clusters. Fidelity is the proportion of events in that cluster with the family recorded. IndVal is specificity × fidelity. These are descriptive indicators and use the untransformed counts as within-event shares.

Among candidates with at least five events per group, prefer the highest final mean silhouette with a distinct, informative leading indicator in every group. A leading indicator must occur in at least half of the group's events, have specificity above 0.5, and have an established family name. If no solution meets those checks, report the highest-silhouette candidate with adequate group sizes and disclose the indicator limitation.

In [4]:
def ward_kmeans(matrix, k):
    ward = AgglomerativeClustering(n_clusters=k, linkage="ward").fit_predict(matrix)
    means = np.vstack([matrix[ward == group].mean(axis=0) for group in range(k)])
    model = KMeans(n_clusters=k, init=means, n_init=1, random_state=seed).fit(matrix)
    return ward, model


def indicator_table(matrix, assignments):
    shares = matrix.div(matrix.sum(axis=1), axis=0)
    averages = shares.groupby(assignments).mean()
    specificity = averages.div(averages.sum(axis=0), axis=1)
    fidelity = matrix.gt(0).groupby(assignments).mean()
    return (
        pd.concat({"specificity": specificity.stack(), "fidelity": fidelity.stack(),
                   "indval": (specificity * fidelity).stack()}, axis=1)
        .rename_axis(["cluster", "taxon"]).reset_index()
        .sort_values(["cluster", "indval", "taxon"], ascending=[True, False, True])
    )


models, silhouettes, indicators, metrics = {}, {}, {}, []
with threadpool_limits(limits=1):
    for k in range(2, max_clusters + 1):
        ward, model = ward_kmeans(features, k)
        sizes = pd.Series(model.labels_).value_counts()
        if len(sizes) != k:
            continue
        sil = silhouette_samples(features, model.labels_)
        table = indicator_table(abundance, model.labels_)
        leading = table.groupby("cluster").head(1)
        informative = (
            leading["specificity"].gt(0.5) & leading["fidelity"].ge(0.5)
            & ~leading["taxon"].str.contains("incertae sedis", case=False)
        )
        models[k], silhouettes[k], indicators[k] = model, sil, table
        metrics.append({
            "model": "Ward + K-Means", "k": k, "clusters": k, "distance": "Euclidean",
            "noise_events": 0, "noise_share": 0.0, "silhouette": sil.mean(),
            "ward_silhouette": silhouette_score(features, ward), "inertia": model.inertia_,
            "smallest_cluster": sizes.min(), "largest_cluster": sizes.max(),
            "evaluated_events": len(features), "informative_indicators": informative.sum(),
            "mean_leading_indval": leading["indval"].mean(),
        })

evaluation = pd.DataFrame(metrics)
eligible = evaluation.loc[evaluation["smallest_cluster"].ge(5)]
if eligible.empty:
    raise ValueError("No tested solution has at least five events in each cluster.")
interpretable = eligible.loc[eligible["informative_indicators"].eq(eligible["k"])]
if interpretable.empty:
    print("No candidate met all indicator criteria. Interpret the leading taxa cautiously.")
    interpretable = eligible
selected_k = int(interpretable.loc[interpretable["silhouette"].idxmax(), "k"])
evaluation["selected"] = evaluation["k"].eq(selected_k)
labels = pd.Series(models[selected_k].labels_, index=abundance.index, name="cluster")
leading_indicators = indicators[selected_k].groupby("cluster").head(1).set_index("cluster")
cluster_quality = (
    pd.DataFrame({"cluster": labels.to_numpy(), "silhouette": silhouettes[selected_k]})
    .groupby("cluster")["silhouette"].agg(events="size", mean="mean", minimum="min", maximum="max")
    .join(leading_indicators)
)
display(evaluation.round(3))
display(cluster_quality.round(3))

,model,k,clusters,distance,noise_events,noise_share,silhouette,ward_silhouette,inertia,smallest_cluster,largest_cluster,evaluated_events,informative_indicators,mean_leading_indval,selected
0,Ward + K-Means,2,2,Euclidean,0,0.0,0.284,0.271,142.997,116,156,272,2,0.748,False
1,Ward + K-Means,3,3,Euclidean,0,0.0,0.271,0.253,117.920,79,111,272,3,0.675,False
2,Ward + K-Means,4,4,Euclidean,0,0.0,0.337,0.320,94.867,41,81,272,3,0.664,False
3,Ward + K-Means,5,5,Euclidean,0,0.0,0.357,0.344,85.572,31,77,272,5,0.656,False
4,Ward + K-Means,6,6,Euclidean,0,0.0,0.369,0.357,78.144,21,77,272,6,0.672,True
5,Ward + K-Means,7,7,Euclidean,0,0.0,0.383,0.374,72.236,15,76,272,6,0.677,False
6,Ward + K-Means,8,8,Euclidean,0,0.0,0.399,0.389,66.414,10,75,272,7,0.687,False


,events,mean,minimum,maximum,taxon,specificity,fidelity,indval
cluster,,,,,,,,
0,32,0.026,-0.186,0.210,Oculinidae,0.875,0.594,0.519
1,26,0.081,-0.180,0.288,Stylasteridae,0.811,0.731,0.592
2,76,0.415,0.010,0.615,Antipathidae,0.629,1.000,0.629
3,40,0.720,0.249,0.844,Plexauridae,0.777,1.000,0.777
4,77,0.388,-0.035,0.593,Ellisellidae,0.665,1.000,0.665
5,21,0.346,-0.087,0.509,Keratoisididae,0.850,1.000,0.850


## Compare with the original family-presence baseline

Reproduce the original K-Means selection on the full binary matrix, then compare labels on shared events. Eta squared is the fraction of variation in recorded family richness explained by the community labels. Cramér's V describes association with the most frequently recorded data provider per event. Neither check isolates the effect of survey effort or validates an ecological mechanism.

Repeat the revised fit at the selected k using only positive counts, a stricter family frequency filter, and, where recorded, sponge categories. Adjusted Rand index (ARI) compares each sensitivity run with the primary labels on shared events. An ARI of 1 indicates identical partitions. DBSCAN is omitted from the revised workflow.

In [5]:
baseline_features = family_presence.to_numpy(dtype="float32")
baseline_max = min(8, len(baseline_features) - 1, len(family_presence.drop_duplicates()))
with threadpool_limits(limits=1):
    baseline_models = [
        KMeans(n_clusters=k, n_init=10, random_state=seed).fit(baseline_features)
        for k in range(2, baseline_max + 1)
    ]
    baseline = max(baseline_models, key=lambda model: silhouette_score(
        baseline_features, model.labels_, sample_size=min(2000, len(baseline_features)), random_state=seed,
    ))
baseline_labels = pd.Series(baseline.labels_, index=family_presence.index, name="baseline_cluster")

def eta_squared(values, groups):
    overall = values.mean()
    summary = values.groupby(groups).agg(["size", "mean"])
    total = values.sub(overall).pow(2).sum()
    return (summary["size"] * summary["mean"].sub(overall).pow(2)).sum() / total if total else np.nan


def provider_association(groups, providers):
    table = pd.crosstab(groups, providers)
    denominator = table.to_numpy().sum() * (min(table.shape) - 1)
    return np.sqrt(chi2_contingency(table, correction=False)[0] / denominator) if denominator else np.nan


richness = family_richness.loc[labels.index]
providers = corals.groupby(event_keys)["data_provider"].agg(
    lambda values: values.mode().iat[0] if values.notna().any() else "Unknown"
).loc[labels.index]
baseline_comparison = pd.DataFrame([
    {"method": name, "events": len(groups), "clusters": groups.nunique(),
     "richness_eta_squared": eta_squared(richness, groups),
     "provider_cramers_v": provider_association(groups, providers)}
    for name, groups in [("Family presence K-Means", baseline_labels.loc[labels.index]),
                         ("Abundance Ward + K-Means", labels)]
])
display(baseline_comparison.round(3))
print(f"ARI versus original baseline: {adjusted_rand_score(baseline_labels.loc[labels.index], labels):.3f}")

sponge_records = matched_records.loc[matched_records["taxon_category"].str.contains("sponge", case=False, na=False)].copy()
for key in event_keys:
    values = sponge_records[key].astype("string").str.strip()
    sponge_records[key] = values.mask(values.str.casefold().isin(missing_identifiers))
sponge_records = sponge_records.dropna(subset=event_keys)
sponge_records["family"] = "Sponge: " + sponge_records["taxon_category"].str.strip()
sensitivity = {
    "Positive counts only": abundance_matrix(corals.loc[corals["count"].gt(0).fillna(False)], minimum_events),
    "Families in at least 10 events": abundance_matrix(corals, 10),
}
if not sponge_records.empty:
    sensitivity["Corals and sponge categories"] = abundance_matrix(pd.concat([corals, sponge_records]), minimum_events)

sensitivity_rows = []
with threadpool_limits(limits=1):
    for name, matrix in sensitivity.items():
        if len(matrix) <= selected_k or len(matrix.drop_duplicates()) < selected_k:
            continue
        x = normalize(matrix.to_numpy(dtype=float), norm="l2")
        _, model = ward_kmeans(x, selected_k)
        alternative = pd.Series(model.labels_, index=matrix.index)
        shared = labels.index.intersection(alternative.index)
        sensitivity_rows.append({
            "variant": name, "events": len(matrix), "taxa": matrix.shape[1], "shared_events": len(shared),
            "ari_on_shared_events": adjusted_rand_score(labels.loc[shared], alternative.loc[shared]) if len(shared) > 1 else np.nan,
        })
display(pd.DataFrame(sensitivity_rows).round(3))

,method,events,clusters,richness_eta_squared,provider_cramers_v
0,Family presence K-Means,272,2,0.212,0.854
1,Abundance Ward + K-Means,272,6,0.153,0.546


ARI versus original baseline: 0.124


,variant,events,taxa,shared_events,ari_on_shared_events
0,Positive counts only,270,29,270,0.615
1,Families in at least 10 events,271,24,271,0.991
2,Corals and sponge categories,274,31,272,0.789


## Export aligned results

Preserve the four original filenames. The binary matrix and family profiles describe family occurrence on the retained events. The chord matrix holds the actual fitting features. Event labels keep the original survey/event keys, the original baseline label for comparisons, and the leading indicator taxon. The legacy DBSCAN label column is nullable because DBSCAN is not fitted in this revision.

In [6]:
family_presence = family_presence.loc[labels.index]
family_profiles = family_presence.groupby(labels).mean().rename_axis(index="cluster", columns=None)
cluster_summary = pd.concat(
    [labels.value_counts().sort_index().rename("events"), family_richness.groupby(labels).mean().rename("mean_families")],
    axis=1,
)
cluster_summary["event_share"] = cluster_summary["events"] / len(labels)
top_families = family_profiles.apply(lambda row: ", ".join(row[row.gt(0)].nlargest(3).index), axis=1)
cluster_summary["most_recorded_families"] = top_families
display(cluster_summary.round(3))

has_location = corals["lat"].between(-90, 90) & corals["lon"].between(-180, 180)
locations = corals.loc[has_location].groupby(event_keys)[["lat", "lon"]].median()
def list_values(series):
    values = {str(value).strip() for value in series.dropna()} - {""}
    return " | ".join(sorted(values))


event_metadata = corals.groupby(event_keys).agg(
    coral_records=("catalog_num", "size"),
    observation_start=("obs_date", "min"),
    observation_end=("obs_date", "max"),
    dataset_ids=("dataset_id", list_values),
    data_providers=("data_provider", list_values),
    sampling_equipment=("sampling_equipment", list_values),
)
event_labels = (
    labels.to_frame().join(baseline_labels).join(family_richness).join(event_metadata).join(locations).reset_index()
)
event_labels["dbscan_cluster"] = pd.array([pd.NA] * len(event_labels), dtype="Int64")
event_labels["indicator_taxon"] = event_labels["cluster"].map(leading_indicators["taxon"])
if event_labels.duplicated(event_keys).any() or len(event_labels) != len(family_presence):
    raise ValueError("Event labels do not align with modeled rows.")

output_dir = data_dir / "processed"
output_dir.mkdir(parents=True, exist_ok=True)
event_labels.to_parquet(output_dir / "taxon_cluster_events.parquet", index=False)
family_presence.reset_index().to_parquet(output_dir / "taxon_cluster_family_presence.parquet", index=False)
evaluation.to_csv(output_dir / "taxon_cluster_metrics.csv", index=False)
family_profiles.to_csv(output_dir / "taxon_cluster_family_profiles.csv")
chord = pd.DataFrame(features, index=abundance.index, columns=abundance.columns)
chord.reset_index().to_parquet(output_dir / "taxon_cluster_chord_features.parquet", index=False)
cluster_quality.to_csv(output_dir / "taxon_cluster_indicators.csv")
print(f"Exported {len(event_labels):,} event labels and {len(evaluation):,} model evaluations.")
print(f"Events with coordinates: {event_labels[['lat', 'lon']].notna().all(axis=1).sum():,}")

,events,mean_families,event_share,most_recorded_families
cluster,,,,
0,32,6.969,0.118,"Ellisellidae, Antipathidae, Madreporidae"
1,26,7.269,0.096,"Stylasteridae, Primnoidae, Caryophylliidae"
2,76,5.526,0.279,"Antipathidae, Ellisellidae, Plexauridae"
3,40,3.275,0.147,"Plexauridae, Antipathidae, Acanthogorgiidae"
4,77,6.364,0.283,"Ellisellidae, Antipathidae, Plexauridae"
5,21,8.143,0.077,"Keratoisididae, Chrysogorgiidae, Coralliidae"


Exported 272 event labels and 7 model evaluations.
Events with coordinates: 272


## Interpret the selected solution

In this NOAA snapshot, 272 of 275 events remained after rare-family filtering. Six clusters had the highest mean silhouette among the candidates with informative leading indicators for every group. The eight-cluster solution separated events more strongly on average, but one leading indicator had specificity below 0.5. Refining Ward's six groups increased the mean silhouette from 0.357 to 0.369. Group means range from 0.026 (Oculinidae) to 0.720 (Plexauridae), so the groups vary substantially in how well they separate.

On the same 272 events, the revised labels accounted for 15.3% of recorded family-richness variation, compared with 21.2% for the original two labels. The association with provider fell from 0.854 to 0.546 (Cramér's V). These reductions support a weaker relationship with richness and provider, with meaningful provider association still present.

The stricter rarity filter yielded nearly the same grouping (ARI 0.991 on shared events). The positive-count-only analysis changed assignments more substantially (ARI 0.615). Adding sponge categories also changed assignments (ARI 0.789). Treat these indicator groups as observed patterns that depend on the abundance proxy and the taxa included.